In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Final year project"
)

print("Project directory:")
print(PROJECT_DIR)

print("\nContents:")

for item in PROJECT_DIR.iterdir():
    print(item.name)


Project directory:
/content/drive/MyDrive/Final year project

Contents:
Spinach_Plant_Health_ConvNeXt_Tiny.ipynb
Spinach Dataset


In [3]:
class_names_expected = {
    "anthracnose",
    "downy mildew",
    "healthy",
    "pest damage",
    "bacterial spot"
}

print("Searching for the spinach dataset...\n")

for folder in PROJECT_DIR.rglob("*"):

    if folder.is_dir():

        child_names = {
            child.name.lower()
            for child in folder.iterdir()
            if child.is_dir()
        }

        if class_names_expected.issubset(child_names):

            print("FOUND DATASET:")
            print(folder)

Searching for the spinach dataset...

FOUND DATASET:
/content/drive/MyDrive/Final year project/Spinach Dataset


In [4]:
from pathlib import Path

DATASET_DIR = Path(
    "/content/drive/MyDrive/Final year project/Spinach Dataset"
)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

print("=" * 60)
print("SPINACH DATASET INSPECTION")
print("=" * 60)

total_images = 0

for class_folder in sorted(DATASET_DIR.iterdir()):

    if not class_folder.is_dir():
        continue

    images = [
        file
        for file in class_folder.rglob("*")
        if file.suffix.lower() in VALID_EXTENSIONS
    ]

    print(
        f"{class_folder.name:20s} : {len(images)} images"
    )

    total_images += len(images)

print("-" * 60)
print(
    f"{'TOTAL':20s} : {total_images} images"
)
print("=" * 60)

SPINACH DATASET INSPECTION
anthracnose          : 783 images
bacterial spot       : 4565 images
downy mildew         : 1783 images
healthy              : 6855 images
pest damage          : 3369 images
------------------------------------------------------------
TOTAL                : 17355 images


In [5]:
from PIL import Image

print("=" * 60)
print("CHECKING IMAGE FILES")
print("=" * 60)

valid_count = 0
invalid_count = 0

for class_folder in sorted(DATASET_DIR.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.rglob("*"):

        if image_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        try:
            with Image.open(image_path) as img:
                img.verify()

            valid_count += 1

        except Exception as e:
            invalid_count += 1
            print("Invalid image:")
            print(image_path)
            print("Error:", e)

print()
print("Valid images  :", valid_count)
print("Invalid images:", invalid_count)

CHECKING IMAGE FILES

Valid images  : 17355
Invalid images: 0


In [6]:
from pathlib import Path
from PIL import Image
import hashlib
from collections import defaultdict

DATASET_DIR = Path(
    "/content/drive/MyDrive/Final year project/Spinach Dataset"
)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def get_file_hash(file_path):
    """Create a hash based on the actual image file contents."""

    hash_object = hashlib.md5()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_object.update(chunk)

    return hash_object.hexdigest()


print("=" * 60)
print("CHECKING FOR EXACT DUPLICATE FILES")
print("=" * 60)

hash_to_files = defaultdict(list)

total_images = 0

for class_folder in sorted(DATASET_DIR.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.rglob("*"):

        if image_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        try:
            file_hash = get_file_hash(image_path)

            hash_to_files[file_hash].append(
                image_path
            )

            total_images += 1

        except Exception as e:

            print(
                f"Could not process: {image_path}"
            )

duplicate_groups = [
    files
    for files in hash_to_files.values()
    if len(files) > 1
]

duplicate_images = sum(
    len(files) - 1
    for files in duplicate_groups
)

print()
print("Total images:", total_images)

print(
    "Duplicate groups:",
    len(duplicate_groups)
)

print(
    "Duplicate images:",
    duplicate_images
)

print("=" * 60)

CHECKING FOR EXACT DUPLICATE FILES

Total images: 17355
Duplicate groups: 188
Duplicate images: 188


In [7]:
from pathlib import Path
import random
import shutil

# ============================================================
# PATHS
# ============================================================

SOURCE_DIR = Path(
    "/content/drive/MyDrive/Final year project/Spinach Dataset"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Final year project/Spinach_Split"
)

# ============================================================
# SETTINGS
# ============================================================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

SEED = 42

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

# Make sure ratios add up to 1
assert TRAIN_RATIO + VAL_RATIO + TEST_RATIO == 1.0

random.seed(SEED)

# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# FIND CLASSES
# ============================================================

class_folders = [
    folder
    for folder in SOURCE_DIR.iterdir()
    if folder.is_dir()
]

print("=" * 70)
print("DATASET SPLITTING")
print("=" * 70)

print("\nClasses found:")

for folder in class_folders:
    print("-", folder.name)

# ============================================================
# SPLIT EACH CLASS
# ============================================================

for class_folder in sorted(class_folders):

    class_name = class_folder.name

    images = [
        file
        for file in class_folder.rglob("*")
        if file.suffix.lower() in VALID_EXTENSIONS
    ]

    # Shuffle images
    random.shuffle(images)

    total = len(images)

    train_count = int(
        total * TRAIN_RATIO
    )

    val_count = int(
        total * VAL_RATIO
    )

    train_images = images[
        :train_count
    ]

    val_images = images[
        train_count:
        train_count + val_count
    ]

    test_images = images[
        train_count + val_count:
    ]

    print("\n" + "-" * 70)

    print(
        f"Class: {class_name}"
    )

    print(
        f"Total: {total}"
    )

    print(
        f"Train: {len(train_images)}"
    )

    print(
        f"Validation: {len(val_images)}"
    )

    print(
        f"Test: {len(test_images)}"
    )

    # ========================================================
    # COPY FILES
    # ========================================================

    split_data = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    for split_name, split_images in split_data.items():

        destination_folder = (
            OUTPUT_DIR /
            split_name /
            class_name
        )

        destination_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        for image_path in split_images:

            destination = (
                destination_folder /
                image_path.name
            )

            shutil.copy2(
                image_path,
                destination
            )

print("\n" + "=" * 70)
print("DATASET SPLIT COMPLETE")
print("=" * 70)

print("\nSaved to:")

print(OUTPUT_DIR)

DATASET SPLITTING

Classes found:
- healthy
- pest damage
- downy mildew
- anthracnose
- bacterial spot

----------------------------------------------------------------------
Class: anthracnose
Total: 783
Train: 548
Validation: 117
Test: 118

----------------------------------------------------------------------
Class: bacterial spot
Total: 4565
Train: 3195
Validation: 684
Test: 686

----------------------------------------------------------------------
Class: downy mildew
Total: 1783
Train: 1248
Validation: 267
Test: 268

----------------------------------------------------------------------
Class: healthy
Total: 6855
Train: 4798
Validation: 1028
Test: 1029

----------------------------------------------------------------------
Class: pest damage
Total: 3369
Train: 2358
Validation: 505
Test: 506

DATASET SPLIT COMPLETE

Saved to:
/content/drive/MyDrive/Final year project/Spinach_Split


In [8]:
import torch
from torchvision import datasets, transforms
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

DATASET_DIR = Path(
    "/content/drive/MyDrive/Final year project/Spinach_Split"
)

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

# ============================================================
# GPU CHECK
# ============================================================

print("=" * 70)
print("SYSTEM CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA version:",
        torch.version.cuda
    )

# ============================================================
# BASIC TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ============================================================
# LOAD DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform
)

# ============================================================
# DISPLAY INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET CHECK")
print("=" * 70)

print("\nClasses:")
print(train_dataset.classes)

print("\nClass → Index:")
print(train_dataset.class_to_idx)

print("\nNumber of training images:", len(train_dataset))
print("Number of validation images:", len(val_dataset))
print("Number of test images:", len(test_dataset))

print("\nExpected classes:", 5)

print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

SYSTEM CHECK
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8

DATASET CHECK

Classes:
['anthracnose', 'bacterial spot', 'downy mildew', 'healthy', 'pest damage']

Class → Index:
{'anthracnose': 0, 'bacterial spot': 1, 'downy mildew': 2, 'healthy': 3, 'pest damage': 4}

Number of training images: 12147
Number of validation images: 2601
Number of test images: 2607

Expected classes: 5

CHECK COMPLETE


#TRAINING



In [10]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

from pathlib import Path
import torch

# Project paths
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Final year project"
)

DATASET_DIR = PROJECT_DIR / "Spinach_Split"

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Training settings
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 5
SEED = 42

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

print("\nDataset:", DATASET_DIR)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)

print("=" * 70)

TRAINING CONFIGURATION
Device: cuda
GPU: Tesla T4
Image size: 224
Batch size: 32
Epochs: 20
Learning rate: 0.0001
Weight decay: 0.0001

Dataset: /content/drive/MyDrive/Final year project/Spinach_Split
Model directory: /content/drive/MyDrive/Final year project/models
Results directory: /content/drive/MyDrive/Final year project/results


In [11]:
# ============================================================
# STEP 1 — DATA TRANSFORMS + DATALOADERS
# ============================================================

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# ImageNet normalization
# ConvNeXt-Tiny is pretrained on ImageNet
# ------------------------------------------------------------

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

# ------------------------------------------------------------
# TRAINING TRANSFORMS
# Augmentation is applied ONLY to training images
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.2
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

# ------------------------------------------------------------
# VALIDATION TRANSFORMS
# No random augmentation
# ------------------------------------------------------------

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

# ------------------------------------------------------------
# TEST TRANSFORMS
# Same deterministic preprocessing as validation
# ------------------------------------------------------------

test_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

# ------------------------------------------------------------
# DATALOADERS
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print("=" * 70)
print("DATALOADER SETUP COMPLETE")
print("=" * 70)

print("\nClasses:")
print(train_dataset.classes)

print("\nTraining images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("\nBatch size:", BATCH_SIZE)
print("Image size:", IMAGE_SIZE)

print("\n" + "=" * 70)

DATALOADER SETUP COMPLETE

Classes:
['anthracnose', 'bacterial spot', 'downy mildew', 'healthy', 'pest damage']

Training images: 12147
Validation images: 2601
Test images: 2607

Training batches: 380
Validation batches: 82
Test batches: 82

Batch size: 32
Image size: 224



In [12]:
# ============================================================
# STEP 2 — CLASS WEIGHTS
# ============================================================

import numpy as np
from collections import Counter

# Get all training labels
train_labels = [
    label
    for _, label in train_dataset.samples
]

# Count images per class
class_counts = Counter(train_labels)

print("=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

for class_index, class_name in enumerate(train_dataset.classes):
    print(
        f"{class_name:20s}: "
        f"{class_counts[class_index]}"
    )

# ------------------------------------------------------------
# Calculate balanced class weights
# ------------------------------------------------------------

total_samples = len(train_labels)
num_classes = len(train_dataset.classes)

class_weights = []

for class_index in range(num_classes):

    count = class_counts[class_index]

    weight = total_samples / (
        num_classes * count
    )

    class_weights.append(weight)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(DEVICE)

print("\n" + "=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

for class_name, weight in zip(
    train_dataset.classes,
    class_weights
):
    print(
        f"{class_name:20s}: "
        f"{weight.item():.4f}"
    )

print("\nWeights device:", class_weights.device)

print("\n" + "=" * 70)

CLASS DISTRIBUTION
anthracnose         : 548
bacterial spot      : 3195
downy mildew        : 1248
healthy             : 4798
pest damage         : 2358

CLASS WEIGHTS
anthracnose         : 4.4332
bacterial spot      : 0.7604
downy mildew        : 1.9466
healthy             : 0.5063
pest damage         : 1.0303

Weights device: cuda:0



In [13]:
# ============================================================
# STEP 3 — LOAD PRETRAINED CONVNEXT-TINY
# ============================================================

import timm
import torch

print("=" * 70)
print("LOADING CONVNEXT-TINY")
print("=" * 70)

# Load pretrained ConvNeXt-Tiny
model = timm.create_model(
    "convnext_tiny.fb_in1k",
    pretrained=True,
    num_classes=NUM_CLASSES
)

# Move model to GPU
model = model.to(DEVICE)

# Count parameters
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel: ConvNeXt-Tiny")
print("Number of classes:", NUM_CLASSES)

print(
    f"Total parameters: {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

print(
    "\nModel device:",
    next(model.parameters()).device
)

print("\n" + "=" * 70)
print("MODEL READY")
print("=" * 70)

LOADING CONVNEXT-TINY


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            


Model: ConvNeXt-Tiny
Number of classes: 5
Total parameters: 27,823,973
Trainable parameters: 27,823,973

Model device: cuda:0

MODEL READY


In [14]:
# ============================================================
# STEP 4 — LOSS FUNCTION + OPTIMIZER
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("SETTING UP TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# Loss function
# ------------------------------------------------------------
# Class weights help compensate for the imbalance in our dataset.
#
# Example:
# Anthracnose has fewer images -> higher weight
# Healthy has many images     -> lower weight
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------
# AdamW updates the model's weights during training.
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# Learning-rate scheduler
# ------------------------------------------------------------
# Gradually reduces the learning rate as training progresses.
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

print("\nLoss function:")
print(criterion)

print("\nOptimizer:")
print(optimizer)

print("\nLearning-rate scheduler:")
print(scheduler)

print("\nInitial learning rate:")
print(optimizer.param_groups[0]["lr"])

print("\n" + "=" * 70)
print("TRAINING COMPONENTS READY")
print("=" * 70)

SETTING UP TRAINING

Loss function:
CrossEntropyLoss()

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)

Learning-rate scheduler:

Initial learning rate:
0.0001

TRAINING COMPONENTS READY


In [15]:
# ============================================================
# STEP 5 — TRAIN CONVNEXT-TINY
# ============================================================

import json
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

print("=" * 70)
print("STARTING CONVNEXT-TINY TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": []
}

# ------------------------------------------------------------
# Best validation score
# ------------------------------------------------------------

best_val_f1 = -1.0
best_epoch = 0

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(NUM_EPOCHS):

    print("\n")
    print("=" * 70)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS}"
    )
    print("=" * 70)

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    train_progress = tqdm(
        train_loader,
        desc="Training",
        leave=True
    )

    for images, labels in train_progress:

        # Move data to GPU
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        # Clear old gradients
        optimizer.zero_grad(
            set_to_none=True
        )

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(
            outputs,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # ----------------------------------------------------
        # Training statistics
        # ----------------------------------------------------

        running_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            (predictions == labels)
            .sum()
            .item()
        )

        total += labels.size(0)

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss / total
    )

    train_accuracy = (
        correct / total
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        val_progress = tqdm(
            val_loader,
            desc="Validation",
            leave=True
        )

        for images, labels in val_progress:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            # Forward pass
            outputs = model(images)

            # Validation loss
            loss = criterion(
                outputs,
                labels
            )

            val_running_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                (predictions == labels)
                .sum()
                .item()
            )

            val_total += labels.size(0)

            # Store predictions for F1
            all_val_labels.extend(
                labels.cpu().numpy()
            )

            all_val_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss = (
        val_running_loss / val_total
    )

    val_accuracy = (
        val_correct / val_total
    )

    val_macro_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    # ========================================================
    # UPDATE LEARNING RATE
    # ========================================================

    scheduler.step()

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    history["val_macro_f1"].append(
        val_macro_f1
    )

    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("\n")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.4f}")
    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy:.4f}")
    print(f"Val Macro F1   : {val_macro_f1:.4f}")
    print(f"Learning Rate  : {current_lr:.8f}")

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_macro_f1 > best_val_f1:

        best_val_f1 = val_macro_f1
        best_epoch = epoch + 1

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_macro_f1": best_val_f1,
            "class_names": train_dataset.classes,
            "image_size": IMAGE_SIZE
        }

        checkpoint_path = (
            MODEL_DIR /
            "best_convnext_tiny.pth"
        )

        torch.save(
            checkpoint,
            checkpoint_path
        )

        print(
            "\n✓ BEST MODEL SAVED"
        )

        print(
            "Path:",
            checkpoint_path
        )

        print(
            f"Best Val Macro F1: "
            f"{best_val_f1:.4f}"
        )

    # ========================================================
    # SAVE TRAINING HISTORY AFTER EVERY EPOCH
    # ========================================================

    history_path = (
        RESULTS_DIR /
        "training_history.json"
    )

    with open(
        history_path,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=4
        )

# ============================================================
# TRAINING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nBest Epoch: {best_epoch}"
)

print(
    f"Best Validation Macro F1: "
    f"{best_val_f1:.4f}"
)

print(
    "\nBest model saved at:"
)

print(
    MODEL_DIR /
    "best_convnext_tiny.pth"
)

print(
    "\nTraining history saved at:"
)

print(
    RESULTS_DIR /
    "training_history.json"
)

print("\n" + "=" * 70)

STARTING CONVNEXT-TINY TRAINING


EPOCH 1/20


Training:   0%|          | 0/380 [00:00<?, ?it/s]

Validation:   0%|          | 0/82 [00:00<?, ?it/s]



Train Loss     : 0.7883
Train Accuracy : 0.6924
Val Loss       : 0.5134
Val Accuracy   : 0.8108
Val Macro F1   : 0.8141
Learning Rate  : 0.00009938

✓ BEST MODEL SAVED
Path: /content/drive/MyDrive/Final year project/models/best_convnext_tiny.pth
Best Val Macro F1: 0.8141


EPOCH 2/20


Training:   0%|          | 0/380 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/82 [00:00<?, ?it/s]



Train Loss     : 0.0062
Train Accuracy : 0.9979
Val Loss       : 0.0836
Val Accuracy   : 0.9769
Val Macro F1   : 0.9809
Learning Rate  : 0.00000062

✓ BEST MODEL SAVED
Path: /content/drive/MyDrive/Final year project/models/best_convnext_tiny.pth
Best Val Macro F1: 0.9809


EPOCH 20/20


Training:   0%|          | 0/380 [00:00<?, ?it/s]

Validation:   0%|          | 0/82 [00:00<?, ?it/s]



Train Loss     : 0.0052
Train Accuracy : 0.9979
Val Loss       : 0.0842
Val Accuracy   : 0.9773
Val Macro F1   : 0.9815
Learning Rate  : 0.00000000

✓ BEST MODEL SAVED
Path: /content/drive/MyDrive/Final year project/models/best_convnext_tiny.pth
Best Val Macro F1: 0.9815


TRAINING COMPLETE

Best Epoch: 20
Best Validation Macro F1: 0.9815

Best model saved at:
/content/drive/MyDrive/Final year project/models/best_convnext_tiny.pth

Training history saved at:
/content/drive/MyDrive/Final year project/results/training_history.json

